# google_cloud_core smoke test

Smoke test for `google-cloud-core` — shared base classes for google-cloud-*
clients. There is no `google_cloud_core` module: the dist installs modules into
the shared `google.cloud` namespace package. A `pkgutil.walk_packages` sweep over
`google.cloud.__path__` would import every sibling google-cloud-* dist, so cell 1
imports exactly the modules this dist owns (from its wheel RECORD) instead.

In [ ]:
# Submodule smoke test (namespace-package variant): import every module the
# google-cloud-core wheel RECORD owns; a full namespace walk would sweep
# sibling google-cloud-* dists.
import importlib

CORE_MODULES = [
    "google.cloud._helpers",
    "google.cloud._http",
    "google.cloud._testing",
    "google.cloud.client",
    "google.cloud.environment_vars",
    "google.cloud.exceptions",
    "google.cloud.obsolete",
    "google.cloud.operation",
    "google.cloud.version",
]

failed = []
for name in CORE_MODULES:
    try:
        importlib.import_module(name)
    except Exception as e:
        failed.append((name, type(e).__name__, str(e)))
        raise

for name, etype, msg in failed:
    print(f"FAIL {name}: {etype}: {msg}")
assert not failed, f"{len(failed)} google-cloud-core module(s) failed to import"
print(f"OK — all {len(CORE_MODULES)} google-cloud-core modules imported cleanly")

In [ ]:
# Example 1: HTTP exception hierarchy — codes + inheritance.
from google.cloud import exceptions

print(f"NotFound.code = {exceptions.NotFound.code}")
print(f"Conflict.code = {exceptions.Conflict.code}")
err = exceptions.NotFound("bucket does not exist")
print(f"str(err) = {err}")
assert exceptions.NotFound.code == 404
assert exceptions.Conflict.code == 409
assert isinstance(err, exceptions.ClientError)
assert isinstance(err, exceptions.GoogleCloudError)

In [ ]:
# Example 2: from_http_status maps status codes to typed exceptions.
from google.cloud import exceptions

nf = exceptions.from_http_status(404, "object missing")
conflict = exceptions.from_http_status(409, "already exists")
generic = exceptions.from_http_status(418, "teapot")

print(f"404 -> {type(nf).__name__}")
print(f"409 -> {type(conflict).__name__}")
print(f"418 -> {type(generic).__name__} (code={generic.code})")
assert isinstance(nf, exceptions.NotFound)
assert isinstance(conflict, exceptions.Conflict)
assert generic.code == 418

In [ ]:
# Example 3: The base Client — constructed offline with anonymous credentials.
from google.auth.credentials import AnonymousCredentials
from google.cloud.client import Client, ClientWithProject

client = Client(credentials=AnonymousCredentials())
print(f"client: {type(client).__name__}")
print(f"credentials: {type(client._credentials).__name__}")
print(f"ClientWithProject is a subclass: {issubclass(ClientWithProject, Client)}")
assert type(client._credentials).__name__ == "AnonymousCredentials"
assert issubclass(ClientWithProject, Client)

In [ ]:
# Example 4: environment_vars constants + version consistency with dist metadata.
import importlib.metadata as md

from google.cloud import environment_vars, version

print(f"GCD_DATASET = {environment_vars.GCD_DATASET}")
print(f"BIGTABLE_EMULATOR = {environment_vars.BIGTABLE_EMULATOR}")
print(f"module version: {version.__version__}")
print(f"dist version:   {md.version('google-cloud-core')}")
assert environment_vars.GCD_DATASET == "DATASTORE_DATASET"
assert version.__version__ == md.version("google-cloud-core")

In [ ]:
# Example 5: obsolete.complain — the deprecation helper other google-cloud libs
# call for renamed dists (it warns only when the named dist is installed).
import warnings

from google.cloud.obsolete import complain

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    complain("google-cloud-core")

print(f"warnings raised: {len(caught)}")
print(f"category: {caught[0].category.__name__}")
print(f"message: {caught[0].message}")
assert len(caught) == 1
assert issubclass(caught[0].category, DeprecationWarning)